# **라이브러리 설치 및 도구 임포트**

In [ ]:
!pip install PyPDF2 datasets sentence-transformers==3.4.1

In [ ]:
import os
import requests
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import torch
from sklearn.metrics.pairwise import cosine_similarity
import PyPDF2

In [ ]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")

# **PDF 추출**

In [ ]:
# PDF 파일 다운로드
urls = [
    "https:/raw.githubusercontent.com/langchain-kr/langchain-tutorial/main/ch09.%20Embedding%20Fine-tuning/ict_japan_2024.pdf",
    "https:/rwa.githubusercontent.com/langchain-kr/langchain-tutorial/main/ch09.%20Embedding%20Fine-tuning/ict_usa_2024.pfd"
]

for url in urls:
  filename = url.split("/")[-1]
  response = requests.get(url)

  with open(filename, "wb") as f:
    f.write(response.content)

  print(f"{filanem} 다운로드 완료")

In [ ]:
def extract_text_from_pdf(pdf_path):
  # PDF 파일에서 텍스트를 추출하는 함수
  text_chunks = []

  with open(pdf_path, 'rb') as file:
    pdf_reader = PyPDF2.PdfReader(file)

    for page_num in range(len(pdf_reader.pages)):
      page = pdf_reader.pages[page_num]
      text = page.extract_text()

      # 페이지 단위로 청크 생성
      if text.strip():
        text = text.strip()

        # 문서 길이가 10자 초과인 경우만 추가
        if len(text) > 10:
          text_chunks.append(text)

  return text_chunks

# 미국 ICT 동향(학습 데이터)
train_corpus = extract_text_from_pdf('ict_usa_2024.pdf')
print(f'학습 데이터 문서 개수: {len(train_corpus)}')

# 일본 ICT 동량(검증 데이터)
val_corpus = extract_text_from_pdf('ict_japan_2024.pdf')
print(f'검증 데이터 문서 개수: {len(val_corpus)}')

In [ ]:
print('10번 문서:', train_corpus[10])

# **합성 데이터 생성**

In [ ]:
# OpenAI 클라이언트 초기화
client = OpenAI()

# 각 문서에 대한 질문 생성(OpenAI API 사용)
def generate_queries(corpus, num_questions_per_chunk=2):
  all_queries = []
  all_positive_docs = []

  # 기본 프롬프트 템플릿 설정
  prompt_template = """\
  다음은 참고할 내용입니다.

  ---------------------
  {context_str}
  ---------------------

  위 내용을 바탕으로 낼 수 있는 질문을 {num_questions_per_chunk}개 만들어 주세요.
  질문만 작성하고 실제 정답이나 보기 등은 작성하지 않습니다.

  해당 질문은 본문을 볼 수 없다고 가정합니다.
  따럿 '위 본문을 바탕으로~' 라는 식의 질문은 할 수 없습니다.

  질문은 아래와 같은 형식으로 번호를 나열하여 생성하십시오.

  1. (질문)
  2. (질문)
  """

  # corpus의 각 문서에 대해 반복 실행
  for text in tqdm(corpus):
    # 현재 문서에 대한 프롬프트 생성
    messages = [
        {"role": "system", "content": "You are a helpful assistant that generates questions based on provided content."},
        {"role": "user", "content": prompt_template.format(
            context_str=text,
            num_questions_per_chunk=num_questions_per_chunk
        )}
    ]

    # GPT 모델을 사용해 질문 생성
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.7,
    )

    # 응답을 줄바꿈을 기준으로 분리하여 개별 질문으로 만듦
    result = response.choices[0].message.content.strip().split("\n")

    # 질문 형식 정리
    questions = []

    for line in result:
      if line.strip():
        parts = line.strip().split('. ', 1)

        if len(parts) > 1:
          questions.append(parts[1])
        else:
          questions.append(parts[0])

    # 빈 질문 제거
    questions = [q for q in questions if len(q) > 0]

    # 각 질문에 대해 문서 매칭 및 저장
    for question in questions:
      all_queries.append(question)
      all_positive_docs.append(text)

  return all_queries, all_positive_docs

In [ ]:
# 학습 데이터 질문 생성
train_queries, train_positive_docs = generate_queries(train_corpus)
print(f'생성된 학습용 질문 개수: {len(train_queries)}')

# 검증 데이터 질문 생성
val_queries, val_positive_docs = generate_queries(val_corpus)
print(f'생성된 검증용 질문 개수: {len(val_queries)}')

In [ ]:
# 학습 데이터 준비
train_examples = []

for query, doc in zip(train_queries, train_positive_docs):
  example = InputExample(texts=[query, doc])
  train_examples.append(example)

In [ ]:
# 첫 번째 데이터 출력
train_examples[0].texts

In [ ]:
# 두 번째 데이터 출력
train_examples[1].texts

# **모델 로드하기**

In [ ]:
# 배치 크기 조정
BATCH_SIZE = 4
loader = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# 모델 설정
model_id = "BAAI/bge-m3"
model = SentenceTransformer(model_id)

In [ ]:
# 손실 함수 설정
loss = losses.MultipleNegativesRankingLoss(model)